# 01 — Data Understanding & Quality Checks

This notebook records the first project stage: understanding the Olist data model, validating data grain, keys, dates, missingness, and usable time coverage.

**Practice goal:** learn to question the data before modeling.


In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv("../data_raw/olist_customers_dataset.csv")
orders = pd.read_csv("../data_raw/olist_orders_dataset.csv")

customers.shape, orders.shape


## Customer identity

- `customer_id`: order/customer-record level identifier.
- `customer_unique_id`: persistent customer identity used for retention, RFM and CLV.

Observed in practice:
- 99,441 customer rows
- 96,096 unique persistent customers
- 2,997 customers appear more than once.


In [ ]:
customers["customer_id"].nunique(), customers["customer_unique_id"].nunique()


In [ ]:
customers["customer_unique_id"].value_counts().value_counts().sort_index()


## Parse order timestamps correctly

The source dates use `DD-MM-YYYY HH:MM`. Using the wrong parser created tens of thousands of false missing timestamps, so the raw format was rechecked before cleaning.


In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        format="%d-%m-%Y %H:%M",
        errors="coerce"
    )


In [ ]:
orders[date_columns].isna().sum()


## Data-quality learning

Missing values are interpreted using the order lifecycle instead of automatically dropping rows.

Examples:
- missing approval timestamps are mostly explainable for cancelled/created orders;
- delivered orders missing delivery timestamps are suspicious and should be excluded only from delivery-duration metrics, not necessarily from revenue/order counts.


In [ ]:
orders.groupby("order_status")["order_delivered_customer_date"].apply(lambda s: s.isna().sum())


## Time coverage

The raw dataset edges are sparse. A working assumption for many analyses is roughly **Jan 2017–Aug 2018**, while keeping right-censoring in mind for retention work.


In [ ]:
monthly_orders = (
    orders.set_index("order_purchase_timestamp")
    .resample("MS")["order_id"]
    .nunique()
)
monthly_orders
